In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

**Install, restart session, repeat**

In [ ]:
#!pip install ydata_profiling

In [ ]:
#!pip install featuretools

In [ ]:
#!pip install autofeat

**Pycaret Lib**

In [ ]:
#pip install "numpy==1.26.4"

In [ ]:
#!python -m pip install git+https://github.com/pycaret/pycaret.git@master --upgrade

In [ ]:
#!pip uninstall wandb -y
#!pip install wandb

## **Load dataset**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/ML AI/AmesHousing.csv')
df.head()

In [ ]:
df.shape

In [ ]:
from ydata_profiling import ProfileReport

profile = ProfileReport(df, title="Ames Housing Data Report")

profile.to_notebook_iframe()

In [ ]:
df.describe()

In [ ]:
# changing the capital letter and replace space with _
df.columns = df.columns.str.lower().str.replace(' ', '_')
df.head()

In [ ]:
# impute missing values

# numerical with median
df['lot_frontage'] = df['lot_frontage'].fillna(df['lot_frontage'].median())

# categorical with none
df['alley'] = df['alley'].fillna('None')

## **Split Train Test**

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['saleprice'])
y = df['saleprice']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

In [ ]:
train_set = X_train.copy()
test_set = X_test.copy()

### **Feature engineering**

#### **1. FeatureTools**

In [ ]:
import featuretools as ft

# Initialize an empty EntitySet
es = ft.EntitySet(id = "ames_housing_data")

# Add the df DataFrame as an entity to the EntitySet
es = es.add_dataframe(dataframe_name="housing", dataframe=train_set, index="order")

# View the tables and the relationships (currently just one table)
print(es)

In [ ]:
# Deep Feature Synthesis (DFS) dengan trans_primitives (karena no relationship?)

feature_matrix_train, feature_defs = ft.dfs(  # ft.dfs kayak FIT di train , feature_defs = hasil belajarnya
    entityset=es,
    target_dataframe_name="housing",
    trans_primitives=["absolute", "num_characters", "num_words"]
)

feature_matrix_train.head()

##### **Test set**

In [ ]:
# buat entityset baru untuk test
es_test = ft.EntitySet(id="ames_housing_test")

# Add X_test as a dataframe to the EntitySet, using the 'order' column as the index
es_test = es_test.add_dataframe(
    dataframe_name="housing",
    dataframe=test_set,
    index="order"
)

In [ ]:
# Filter feature_defs to exclude features that reference 'saleprice'
# This is a workaround as feature_defs should ideally be generated from X_train to begin with.
filtered_feature_defs = [f for f in feature_defs if 'saleprice' not in str(f).lower()]

# Calculate feature matrix for the test set using the filtered feature definitions
feature_matrix_test = ft.calculate_feature_matrix(filtered_feature_defs, entityset=es_test)

feature_matrix_test.head()

In [ ]:
# take only numeric column, to check amount of columns
train_ft= X_train.select_dtypes(include='number')
test_ft= X_test.select_dtypes(include='number')

print(train_ft.shape)
print(test_ft.shape)

#### **2. Autofeat**

In [ ]:
train_auto = X_train.copy()
test_auto = X_test.copy()

In [ ]:
# one-hot encoding
train_ohe = pd.get_dummies(train_auto, drop_first= True)
test_ohe = test_auto.reindex(
    columns = train_ohe.columns,
    fill_value = 0)

In [ ]:
# Check for remaining NaNs in train_ohe before the fillna(0) step
nan_counts = train_ohe.isnull().sum()
columns_with_nan = nan_counts[nan_counts > 0]

if not columns_with_nan.empty:
    print("Columns in `train_ohe` that still contained NaN values:")
    print(columns_with_nan)
else:
    print("No NaN values found in `train_ohe` before the fillna(0) step.")

In [ ]:
# only in train set

from autofeat import AutoFeatRegressor

afrog = AutoFeatRegressor(
    featsel_runs=2,
    n_jobs=1,
    verbose=-1)

# Impute NaN values in train_ohe and test_ohe before passing to AutoFeatRegressor
# fill with 0 to ensure no NaNs are present.
train_ohe_filled = train_ohe.fillna(0)
test_ohe_filled = test_ohe.fillna(0)

train_af_features = afrog.fit_transform(train_ohe_filled, y_train)
test_af_features = afrog.transform(test_ohe_filled)

# Add saleprice to the Autofeat engineered feature dataframes for PyCaret
train_af = train_af_features.copy()
train_af['saleprice'] = y_train.values

test_af = test_af_features.copy()
test_af['saleprice'] = y_test.values

In [ ]:
print("Before AutoFeat:", train_ohe.shape)
print("After AutoFeat :", train_af.shape)


## **Pycaret to choose the best dataset (RMSE)**

In [ ]:
# baseline

train_baseline = X_train.copy()
train_baseline['saleprice'] = y_train

test_baseline = X_test.copy()
test_baseline['saleprice'] = y_test

In [ ]:
# FeatureTools

# train
# Make 'order' a regular column in feature_matrix_train for explicit merging
train_ft = feature_matrix_train.reset_index(names='order')
# Create a temporary DataFrame from y_train and X_train['order'] for merging
y_train_with_order = pd.DataFrame({'order': X_train['order'].values, 'saleprice': y_train.values})
# Merge train_ft with y_train_with_order on the 'order' column
train_ft = pd.merge(train_ft, y_train_with_order, on='order', how='left')

# test
test_ft = feature_matrix_test.reset_index(names='order')
y_test_with_order = pd.DataFrame({'order': X_test['order'].values, 'saleprice': y_test.values})
test_ft = pd.merge(test_ft, y_test_with_order, on='order', how='left')

In [ ]:
# Autofeat

# train
train_auto = X_train.copy()
train_auto['saleprice'] = y_train

# test
test_auto = X_test.copy()
test_auto['saleprice'] = y_test

In [ ]:
from pycaret.regression import RegressionExperiment
from sklearn.metrics import mean_squared_error

def run_one(train_df, test_df):
    exp = RegressionExperiment()

    exp.setup(
        data=train_df,
        target='saleprice',
        session_id=42,
        fold=2,
        normalize=False,
        verbose=False
    )

    best = exp.compare_models(sort='RMSE')
    preds = exp.predict_model(best, data=test_df)

    rmse = mean_squared_error(
        preds['saleprice'],
        preds['prediction_label'],
        squared=False
    )

    return best, rmse


In [ ]:
best_base, rmse_base = run_one(train_baseline, test_baseline)
best_ft, rmse_ft     = run_one(train_ft, test_ft)
best_af, rmse_af     = run_one(train_af, test_af)

print("RMSE TEST Baseline     :", rmse_base)
print("RMSE TEST FeatureTools :", rmse_ft)
print("RMSE TEST AutoFeat     :", rmse_af)

In [ ]:
def evaluate_models(train_df, test_df):
    exp = RegressionExperiment()

    exp.setup(
        data=train_df,
        target='saleprice',
        session_id=42,
        fold=5,
        normalize=False,
        verbose=False
    )

    models = {
        "Linear Regression": "lr",
        "Random Forest": "rf",
        "Gradient Boosting": "gbr"
    }

    results = {}

    for name, code in models.items():
        m = exp.create_model(code)
        preds = exp.predict_model(m, data=test_df)

        rmse = mean_squared_error(
            preds['saleprice'],
            preds['prediction_label'],
            squared=False
        )

        results[name] = rmse

    return results


In [ ]:
results = evaluate_models(train_baseline, test_baseline)
print(results)

In [ ]:
# 1. Choose the best result from run_one
best_result = train_ft

# 2. Re-setup globally (outside of function)
from pycaret.regression import RegressionExperiment
exp_final = RegressionExperiment()
exp_final.setup(data=best_result, target='saleprice', session_id=42)

# 3. Final model
model_final = exp_final.create_model('gbr')

In [ ]:
#  feature importance

exp_final.plot_model(model_final, plot='feature')

In [ ]:
# save feature plot

# exp_final.plot_model(model_final, plot='feature', save=True)

# from google.colab import files
# files.download('Feature Importance.png')

In [ ]:
exp_final.plot_model(model_final, plot='error')

In [ ]:
# save error plot

# exp_final.plot_model(model_final, plot='error', save=True)

# from google.colab import files
# files.download('Prediction Error.png')